In [1]:
%cd ../../

/home/hoanghu/projects/fw-models


In [2]:
from itertools import product

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import polars as pl
from scipy.spatial import distance
from sklearn.preprocessing import TargetEncoder, OrdinalEncoder
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from xgboost import XGBRegressor
from catboost import CatBoostRegressor
from lightgbm import LGBMRegressor
from sklearn.metrics import root_mean_squared_error, r2_score

In [3]:
plt.style.use('seaborn-v0_8')
plt.rcParams.update({'font.size': 8})

# Load dims and raw data

## dim `meals`

In [4]:
meals = pl.read_excel("data/processed/phase_4/dim_meals.xlsx")
meals.head()

meal_id,meal_type_1,schoolyear,is_kela,is_new,restaurant,meal_type_2,pcs_mean
i64,str,str,bool,bool,str,str,f64
9017,"""vegan""","""24-25""",true,true,"""che-exa-vik""","""vegan-miscellaneous""",116.3425
7201,"""vegan""","""23-24""",false,false,null,null,106.231119
9032,"""vegan""","""23-24""",false,false,null,null,106.231119
9102,"""vegan""","""23-24""",false,false,null,null,106.231119
7010,"""vegetarian""","""24-25""",false,false,"""che-exa-vik""",null,71.0


In [18]:
meal_types = meals.select('meal_id', pl.col('meal_type_1').alias('meal_type'))

## dim `meal_names`

In [5]:
meal_names = pl.read_excel("data/processed/phase_4/dim_meal_names.xlsx")
meal_names.head()

meal_id,meal
i64,str
9017,"""""Butter"" härkäpapua & pähkinää"""
7201,"""2023 Härkäpu-sienilasagnette"""
9032,"""Appelisiini-luomukikhernecurry…"
9102,"""Artisokkavugetteja & tuoretoma…"
7010,"""Aurajuusto-pinaattilasagnette"""


## dim `opentime`

In [6]:
dim_opentime = (
    pl
    .from_records([
        {'restaurant': 'che', 'time_open': '10:30', 'time_close': '15:00'},
        {'restaurant': 'exa', 'time_open': '11:00', 'time_close': '14:00'},
        {'restaurant': 'phy', 'time_open': '10:00', 'time_close': '15:00'},
        {'restaurant': 'vik', 'time_open': '10:30', 'time_close': '14:00'},
    ])
    .select(
        'restaurant',
        pl.col('time_open').str.to_datetime("%H:%M"),
        pl.col('time_close').str.to_datetime("%H:%M")
    )
    .with_columns(
        ((pl.col('time_close') - pl.col('time_open')).dt.total_minutes() / 60.).alias('working_duration')
    )
)

dim_opentime.head()

restaurant,time_open,time_close,working_duration
str,datetime[μs],datetime[μs],f64
"""che""",0001-01-01 10:30:00,0001-01-01 15:00:00,4.5
"""exa""",0001-01-01 11:00:00,0001-01-01 14:00:00,3.0
"""phy""",0001-01-01 10:00:00,0001-01-01 15:00:00,5.0
"""vik""",0001-01-01 10:30:00,0001-01-01 14:00:00,3.5


## dim meal names' embedding

In [7]:
meal_embds = pl.read_parquet("data/inter/meal_names_embds.parquet").drop('__index_level_0__', 'meal')
meal_embds.head()

meal_id,embedding
i64,list[f64]
7010,"[0.011678, -0.039025, … 0.035513]"
7010,"[0.01346, -0.047018, … 0.038673]"
1751,"[0.050395, 0.037269, … 0.025638]"
1751,"[0.065942, 0.037054, … 0.022381]"
200006,"[0.020469, 0.026279, … 0.028041]"


In [8]:
meal_embds = meal_embds.group_by('meal_id').first()

In [9]:
meal_sims = meal_embds.join(meal_embds, how='cross')

embds_1 = meal_sims.get_column('embedding').to_numpy()
embds_2 = meal_sims.get_column('embedding_right').to_numpy()
dist = [distance.cosine(e1, e2) for e1, e2 in zip(embds_1, embds_2)]

meal_sims = meal_sims.with_columns(pl.Series(dist).alias('dist')).drop('embedding', 'embedding_right')

meal_sims.head()

meal_id,meal_id_right,dist
i64,i64,f64
9500031,9500031,0.0
9500031,9500027,0.48703
9500031,7000,0.453283
9500031,3299,0.393246
9500031,927,0.501288


## raw POS

In [10]:
paths = [
    "data/raw/pos/Sold lunches.csv",
    "data/raw/pos/Sold lunches Kumpula 6-8 2024.csv",
    "data/raw/pos/Sold lunches Kumpula 9-10 2024.csv",
    "data/raw/pos/Sold lunches Viikuna 2023.csv",
    "data/raw/pos/Sold lunches Viikuna 2024.csv"
]

raw = []
for path in paths:
    df = pl.read_csv(path, separator=';', has_header=False, skip_rows=1)
    # df.columns = np.arange(df.shape[1], dtype=str)
    raw.append(df)


pos = pl.concat(raw)
pos.head()

column_1,column_2,column_3,column_4,column_5,column_6,column_7
str,str,str,str,str,i64,str
"""2.1.2023""","""10:31""","""600 Chemicum""","""Liha""","""Uunimakkaraa,sinappikastiketta""",1,"""0,9"""
"""2.1.2023""","""10:32""","""600 Chemicum""","""Kala""","""Kalapuikot tillikermaviilikast""",1,"""1,04"""
"""2.1.2023""","""10:32""","""600 Chemicum""","""Liha""","""Uunimakkaraa,sinappikastiketta""",1,"""0,9"""
"""2.1.2023""","""10:35""","""600 Chemicum""","""Kala""","""Kalapuikot tillikermaviilikast""",1,"""1,04"""
"""2.1.2023""","""10:36""","""600 Chemicum""","""Liha""","""Uunimakkaraa,sinappikastiketta""",2,"""1,8"""


In [11]:
# Rename columns
pos.columns = ['date', 'time', 'restaurant', 'meal_type', 'meal', 'pcs', 'co2']



# Convert pcs
pos = pos.filter((pl.col('pcs').is_not_null()) & (pl.col('pcs') >= 0))


# Map restaurant name
names_restaurant = {
    '600 Chemicum': 'che', #'chemicum',
    '610 Physicum': 'phy', #'physicum',
    '620 Exactum': 'exa', #'exactum'
    '570 Viikuna': 'vik',
}
pos = pos.with_columns(pl.col('restaurant').replace_strict(names_restaurant))


# Process date
pos = (
    pos
    .with_columns(
        (pl.col('date') + " " + pl.col('time')).str.to_datetime("%d.%m.%Y %H:%M").alias('datetime')
    )
    .drop('date', 'time')
)


# Process meal
pos = (
    pos

    # Remove trailing spaces
    .with_columns(
        pl.col('meal').str.strip_chars(' ')
    )

    
    # Get meal_id
    .join(meal_names, on='meal', how='left')

    
    # Remove entries having no `meal_id`
    .filter(pl.col('meal_id').is_not_null())
)



# Aggregate by date and supplement serving duration per day
pos = (
    pos
    .group_by('restaurant', pl.col('datetime').dt.date().alias('date'), 'meal_id')
    .agg(
        pl.col('pcs').sum(),
        pl.col('datetime').min().alias('time_start'),
        pl.col('datetime').max().alias('time_end'),
    )
    .with_columns(
        ((pl.col('time_end') - pl.col('time_start')).dt.total_minutes() / 60.).alias('serving_duration')
    )
    .join(
        dim_opentime.select('restaurant', 'working_duration'),
        on='restaurant',
        how='left'
    )
    .with_columns(
        (pl.col('serving_duration') / pl.col('working_duration')).alias('serving_percent')
    )
)



# Remove redundant columns
pos = pos.drop('time_start', 'time_end', 'serving_duration', 'working_duration')


# Add meal_type
meal_types = meals.select(
    'meal_id',
    pl.col('meal_type_1').alias('meal_type')
)
pos = pos.join(meal_types, on='meal_id', how='left')



# Ignore buffet
pos = pos.filter(pl.col('meal_type') != pl.lit('buffet'))


pos.head()

restaurant,date,meal_id,pcs,serving_percent,meal_type
str,date,i64,i64,f64,str
"""phy""",2023-11-10,3012,1,0.0,"""vegan"""
"""phy""",2023-05-03,916,14,1.193333,"""fish"""
"""phy""",2023-04-05,1512,16,0.866667,"""vegetarian"""
"""che""",2023-11-29,9500041,60,0.97037,"""fish"""
"""che""",2024-06-19,6670,107,0.562963,"""vegan"""


# Prepare data

## Refer every meals to top K highest sales meals per restaurant and meal_type

In [12]:
CUTOFF_DATE = pl.lit("2024-09-01", dtype=pl.Date)
pos_train = pos.filter(pl.col('date') < CUTOFF_DATE)

In [20]:
K = 10
topK = (
    pos_train
    .group_by('restaurant', 'meal_type', 'meal_id')
    .agg(pl.col('pcs').sum())
    .with_columns(
        pl.col('pcs').rank(descending=True).over('restaurant', 'meal_type').alias('rank')
    )
    .filter(pl.col('rank') <= K)
    .drop('rank', 'pcs')
)


pcs_mean = (
    pos_train
    .group_by('restaurant', 'meal_type', 'meal_id')
    .agg(pl.col('pcs').mean().alias('pcs_mean'))
)
topK = (
    topK
    .join(pcs_mean, on=['restaurant', 'meal_type', 'meal_id'], how='left')
)



# Add embedding
topK = topK.join(meal_embds, on='meal_id')



topK.head()

restaurant,meal_type,meal_id,pcs_mean,embedding
str,str,i64,f64,list[f64]
"""vik""","""chicken""",9500027,62.421053,"[0.020206, -0.002054, … -0.004274]"
"""exa""","""chicken""",9500027,128.153846,"[0.020206, -0.002054, … -0.004274]"
"""exa""","""vegan""",927,55.043478,"[0.103983, -0.054888, … 0.012939]"
"""vik""","""chicken""",6005,67.25,"[0.012241, 0.04705, … 0.008522]"
"""exa""","""chicken""",6005,113.0,"[0.012241, 0.04705, … 0.008522]"


In [21]:
meal_sims = meal_sims.join(
    topK.select(pl.col('meal_id').unique()),
    left_on='meal_id_right',
    right_on='meal_id',
    how='inner'
)

meal_sims.head()

meal_id,meal_id_right,dist
i64,i64,f64
9500031,9500027,0.48703
9500031,927,0.501288
9500031,6005,0.547131
9500031,9500022,0.364582
9500031,9500030,0.439504


In [22]:
pos.head()

restaurant,date,meal_id,pcs,serving_percent,meal_type
str,date,i64,i64,f64,str
"""phy""",2023-11-10,3012,1,0.0,"""vegan"""
"""phy""",2023-05-03,916,14,1.193333,"""fish"""
"""phy""",2023-04-05,1512,16,0.866667,"""vegetarian"""
"""che""",2023-11-29,9500041,60,0.97037,"""fish"""
"""che""",2024-06-19,6670,107,0.562963,"""vegan"""
